# 12 — Provision the Basic Hydro Telemetry Real-Time Dashboard

Deploys **`RTI_Hydro_Telemetry_Basic`**, a Real-Time Dashboard with a *Station* and *Turbine*
filter plus one time chart per sensor group (Power, Pressure, Speed, Temperature, Vibration).

**The dashboard definition is a file, not code.** To redesign: edit the dashboard in the Fabric
UI, `Manage → Download file`, drop the JSON into `Files/dashboards/`, and re-run this notebook.
The embedded seed in CELL 3 is only used when no file is present yet.

**Filters are populated from data.** `OPCUAEvents` only carries `opcua_node_id`, so the
station / turbine / sensor-group hierarchy is joined live from the Lakehouse silver tables:

| Eventhouse object | Purpose |
| --- | --- |
| `silver_instruments`, `silver_equipment`, `silver_facilities` | OneLake shortcuts (delta external tables) over the Lakehouse |
| `AssetMaster()` | `opcua_node_id` -> Station / Turbine / Signal / SignalGroup / Unit |
| `TelemetryEnriched(start, end, stations, turbines)` | `OPCUAEvents` joined to `AssetMaster()` and filtered by the dashboard parameters |

Nothing is hard coded: adding a facility or turbine to the STID source files makes it appear in
the filters after the next medallion run.

Runs standalone, and as `NB12_basicdash` in the `RTI_Orchestrator_Setup` DAG (after NB02 + NB03).

> Generated by `Raw/RTI_Notebooks/tools/build_rti_012.py` — edit that script, not this notebook.


In [ ]:
# =========================
# CELL 0 - Configuration
# Leave the overrides blank to auto-resolve from rti_demo_settings / Fabric REST.
# =========================

DASHBOARD_NAME = "RTI_Hydro_Telemetry_Basic"

# Definition file is read from, in priority order:
#   1. <lakehouse>/Files/dashboards/<DASHBOARD_NAME>.json   (your downloaded redesign)
#   2. DASHBOARD_SOURCE_URL                                 (optional, off by default)
#   3. DASHBOARD_SEED_JSON embedded in CELL 3               (fresh workspace bootstrap)
DASHBOARD_FILES_DIR = "Files/dashboards"
DASHBOARD_SOURCE_URL = ""

# Overrides (blank = auto-resolve)
WORKSPACE_ID = ""
CLUSTER_QUERY_URI = ""
KQL_DB_ID = ""
KQL_DB_NAME = ""
LAKEHOUSE_ID = ""
LAKEHOUSE_NAME = ""
TARGET_FOLDER_ID = ""

# Lakehouse dimension tables that make the filters data driven.
SILVER_INSTRUMENTS = "silver_instruments"
SILVER_EQUIPMENT = "silver_equipment"
SILVER_FACILITIES = "silver_facilities"

SETTINGS_TABLE = "rti_demo_settings"
FABRIC_BASE_URL = "https://api.fabric.microsoft.com/v1"

print("Dashboard name:", DASHBOARD_NAME)


In [ ]:
# =========================
# CELL 1 - Resolve settings and acquire tokens
# =========================

import json
import time
import uuid
import base64

import requests
import notebookutils


def _clean(value):
    return str(value).strip() if value is not None and str(value).strip() else None


# rti_demo_settings is the preferred source, but the notebook must also run standalone.
settings = {}
try:
    spark.catalog.clearCache()
    spark.sql(f"REFRESH TABLE {SETTINGS_TABLE}")
    settings = {r["setting_name"]: r["setting_value"] for r in spark.read.table(SETTINGS_TABLE).collect()}
    print(f"Loaded {len(settings)} rows from {SETTINGS_TABLE}.")
except Exception as exc:  # noqa: BLE001 - standalone run without the lakehouse attached
    print("Could not read", SETTINGS_TABLE, "-", exc)
    print("Falling back to Fabric REST discovery.")


def setting(*names):
    for name in names:
        value = _clean(settings.get(name))
        if value:
            return value
    return None


def _spn_token(scope):
    vault = setting("key_vault_uri")
    if not vault:
        raise RuntimeError("No token available and no key_vault_uri in settings.")
    tenant = notebookutils.credentials.getSecret(vault, setting("key_vault_tenant_id_secret") or "tenantid")
    client = notebookutils.credentials.getSecret(vault, setting("key_vault_client_id_secret") or "clientid")
    secret = notebookutils.credentials.getSecret(vault, setting("key_vault_client_secret_secret") or "clientsecret")
    resp = requests.post(
        f"https://login.microsoftonline.com/{tenant}/oauth2/v2.0/token",
        data={"client_id": client, "client_secret": secret, "grant_type": "client_credentials", "scope": scope},
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


def token_for(audience, scope):
    try:
        token = notebookutils.credentials.getToken(audience)
        if token:
            print(f"{audience} token: caller identity.")
            return token
    except Exception as exc:  # noqa: BLE001
        print(f"getToken('{audience}') unavailable:", exc)
    print(f"{audience} token: Key Vault SPN.")
    return _spn_token(scope)


FABRIC_TOKEN = token_for("pbi", "https://api.fabric.microsoft.com/.default")
FABRIC_HEADERS = {"Authorization": f"Bearer {FABRIC_TOKEN}", "Content-Type": "application/json"}


def list_items(workspace_id):
    items, url = [], f"{FABRIC_BASE_URL}/workspaces/{workspace_id}/items"
    while url:
        resp = requests.get(url, headers=FABRIC_HEADERS)
        resp.raise_for_status()
        body = resp.json()
        items.extend(body.get("value", []))
        url = body.get("continuationUri")
    return items


workspace_id = _clean(WORKSPACE_ID) or setting("workspace_id") or notebookutils.runtime.context.get("currentWorkspaceId")
if not workspace_id:
    raise RuntimeError("Could not determine the workspace id. Set WORKSPACE_ID in CELL 0.")

items = list_items(workspace_id)


def find_item(item_type, name=None):
    for item in items:
        if item.get("type", "").lower() != item_type.lower():
            continue
        if name and item.get("displayName") != name:
            continue
        return item
    return None


cluster_query_uri = _clean(CLUSTER_QUERY_URI) or setting("cluster_query_uri")
kql_db_id = _clean(KQL_DB_ID) or setting("fabric_kql_db_id", "kql_database_id")
kql_db_name = _clean(KQL_DB_NAME) or setting("fabric_kql_db_name", "kql_database_name")
target_folder_id = _clean(TARGET_FOLDER_ID) or setting("target_folder_id")

eventhouse = find_item("Eventhouse")
if eventhouse and (not cluster_query_uri or not kql_db_id):
    resp = requests.get(f"{FABRIC_BASE_URL}/workspaces/{workspace_id}/eventhouses/{eventhouse['id']}", headers=FABRIC_HEADERS)
    resp.raise_for_status()
    props = resp.json().get("properties", {})
    cluster_query_uri = cluster_query_uri or props.get("queryServiceUri")
    db_ids = props.get("databasesItemIds") or []
    kql_db_id = kql_db_id or (db_ids[0] if db_ids else None)
    kql_db_name = kql_db_name or eventhouse.get("displayName")
    target_folder_id = target_folder_id or eventhouse.get("folderId")

# The workspace holds more than one lakehouse (the ontology item creates its own), so never
# just take the first one - resolve by id, then by name, and only then guess.
lakehouse_id = _clean(LAKEHOUSE_ID) or setting("lakehouse_id", "fabric_lakehouse_id")
if not lakehouse_id:
    wanted = _clean(LAKEHOUSE_NAME) or setting("lakehouse_name")
    match = find_item("Lakehouse", wanted) if wanted else None
    if not match:
        candidates = [i for i in items if i.get("type", "").lower() == "lakehouse" and "_lh_" not in i.get("displayName", "")]
        match = candidates[0] if candidates else find_item("Lakehouse")
    if not match:
        raise RuntimeError("No lakehouse found. Set LAKEHOUSE_ID or LAKEHOUSE_NAME in CELL 0.")
    lakehouse_id = match["id"]
    print("Resolved lakehouse by name:", match["displayName"])

missing = [n for n, v in {
    "cluster_query_uri": cluster_query_uri,
    "kql_db_id": kql_db_id,
    "kql_db_name": kql_db_name,
    "lakehouse_id": lakehouse_id,
}.items() if not v]
if missing:
    raise RuntimeError(f"Unresolved settings: {missing}. Fill them in CELL 0.")

cluster_query_uri = cluster_query_uri.rstrip("/")
KUSTO_TOKEN = token_for("kusto", f"{cluster_query_uri}/.default")


def kusto(csl, endpoint="mgmt"):
    resp = requests.post(
        f"{cluster_query_uri}/v1/rest/{endpoint}",
        headers={"Authorization": f"Bearer {KUSTO_TOKEN}", "Content-Type": "application/json"},
        json={"db": kql_db_name, "csl": csl},
    )
    if resp.status_code != 200:
        raise RuntimeError(f"Kusto {endpoint} failed ({resp.status_code}): {resp.text[:800]}")
    tables = resp.json()["Tables"]
    table = next((t for t in tables if t["TableName"] == "Table_0"), tables[0])
    return [c["ColumnName"] for c in table["Columns"]], table["Rows"]


print("Workspace     :", workspace_id)
print("Cluster       :", cluster_query_uri)
print("KQL database  :", kql_db_name, kql_db_id)
print("Lakehouse     :", lakehouse_id)
print("Target folder :", target_folder_id or "(workspace root)")


In [ ]:
# =========================
# CELL 2 - Provision the data-driven asset mapping in the Eventhouse
# OneLake shortcuts over the Lakehouse silver tables + two KQL functions.
# Idempotent: safe to re-run.
# =========================

ONELAKE_ROOT = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables"

for table in (SILVER_INSTRUMENTS, SILVER_EQUIPMENT, SILVER_FACILITIES):
    kusto(
        f".create-or-alter external table {table} kind=delta\n"
        f"(\n  h@'{ONELAKE_ROOT}/{table};impersonate'\n)"
    )
    print(f"shortcut ready: {table}")

kusto(f""".create-or-alter function
with (docstring='Asset master: OPC UA node -> station / turbine / sensor group, joined live from the Lakehouse silver shortcuts.', folder='RTI')
AssetMaster() {{
    external_table('{SILVER_INSTRUMENTS}')
    | project opcua_node_id, Signal = tag, equipment_id, facility_id, Unit = unit, SignalGroup = instrument_type
    | join kind=inner (external_table('{SILVER_EQUIPMENT}') | project equipment_id, Turbine = tag) on equipment_id
    | join kind=inner (external_table('{SILVER_FACILITIES}') | project facility_id, Station = facility_name) on facility_id
    | project opcua_node_id, Station, Turbine, Signal, SignalGroup, Unit
}}""")
print("function ready: AssetMaster()")

kusto(""".create-or-alter function
with (docstring='OPC UA telemetry enriched with asset master and filtered by the dashboard station/turbine parameters.', folder='RTI')
TelemetryEnriched(startTime:datetime, endTime:datetime, stations:dynamic, turbines:dynamic) {
    OPCUAEvents
    | where event_time between (startTime .. endTime)
    | lookup kind=inner AssetMaster() on opcua_node_id
    | where Station in (stations) or isempty(stations)
    | where Turbine in (turbines) or isempty(turbines)
    | project event_time, Station, Turbine, Signal, SignalGroup, Unit, value, quality
}""")
print("function ready: TelemetryEnriched()")

cols, rows = kusto("AssetMaster() | summarize Nodes=count(), Stations=dcount(Station), Turbines=dcount(Turbine), Groups=make_set(SignalGroup)", endpoint="query")
print("\nasset master ->", dict(zip(cols, rows[0])))

cols, rows = kusto("AssetMaster() | distinct Station | sort by Station asc", endpoint="query")
print("station filter options ->", [r[0] for r in rows])

cols, rows = kusto("AssetMaster() | distinct Turbine | sort by Turbine asc", endpoint="query")
print("turbine filter options ->", [r[0] for r in rows])


In [ ]:
# =========================
# CELL 3 - Embedded seed definition (GENERATED - do not hand edit)
# Mirrors Raw/RTI_Notebooks/dashboards/RTI_Hydro_Telemetry_Basic.json.
# Only used when Files/dashboards/ has no copy yet; regenerate with
# python Raw/RTI_Notebooks/tools/build_rti_012.py
# =========================

DASHBOARD_SEED_JSON = r"""{
  "$schema": "https://dataexplorer.azure.com/static/d/schema/77/dashboard.json",
  "schema_version": 77,
  "title": "RTI Hydro Telemetry (Basic)",
  "flavor": "RTDashboard_Regular",
  "autoRefresh": {
    "enabled": true,
    "defaultInterval": "1m",
    "minInterval": "30s"
  },
  "baseQueries": [],
  "embeddedApps": [],
  "dataSources": [
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000001",
      "kind": "kusto-trident",
      "clusterUri": "__CLUSTER_QUERY_URI__",
      "databaseArtifactId": "__KQL_DB_ID__",
      "database": "__KQL_DB_NAME__",
      "workspace": "__WORKSPACE_ID__",
      "name": "__KQL_DB_NAME__"
    }
  ],
  "pages": [
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000002",
      "name": "Telemetry"
    }
  ],
  "parameters": [
    {
      "kind": "duration",
      "id": "a1b2c3d4-0000-4000-8000-000000000003",
      "displayName": "Time range",
      "description": "Time window applied to every tile.",
      "beginVariableName": "_startTime",
      "endVariableName": "_endTime",
      "defaultValue": {
        "kind": "dynamic",
        "count": 4,
        "unit": "hours"
      },
      "showOnPages": {
        "kind": "all"
      }
    },
    {
      "kind": "string",
      "id": "a1b2c3d4-0000-4000-8000-000000000004",
      "displayName": "Station",
      "description": "Populated from silver_facilities through the AssetMaster() function.",
      "variableName": "_station",
      "selectionType": "array",
      "includeAllOption": true,
      "allIsNull": true,
      "defaultValue": {
        "kind": "all"
      },
      "showOnPages": {
        "kind": "all"
      },
      "dataSource": {
        "kind": "query",
        "queryRef": {
          "kind": "query",
          "queryId": "a1b2c3d4-0000-4000-8000-000000000011"
        },
        "columns": {
          "value": "Station"
        }
      }
    },
    {
      "kind": "string",
      "id": "a1b2c3d4-0000-4000-8000-000000000005",
      "displayName": "Turbine",
      "description": "Populated from silver_equipment through the AssetMaster() function, narrowed by the selected stations.",
      "variableName": "_turbine",
      "selectionType": "array",
      "includeAllOption": true,
      "allIsNull": true,
      "defaultValue": {
        "kind": "all"
      },
      "showOnPages": {
        "kind": "all"
      },
      "dataSource": {
        "kind": "query",
        "autoReset": true,
        "queryRef": {
          "kind": "query",
          "queryId": "a1b2c3d4-0000-4000-8000-000000000012"
        },
        "columns": {
          "value": "Turbine"
        }
      }
    }
  ],
  "queries": [
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000011",
      "dataSource": {
        "kind": "inline",
        "dataSourceId": "a1b2c3d4-0000-4000-8000-000000000001"
      },
      "usedVariables": [],
      "text": "// Station options come from the Lakehouse silver tables, never from a hard coded list.\nAssetMaster()\n| distinct Station\n| sort by Station asc"
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000012",
      "dataSource": {
        "kind": "inline",
        "dataSourceId": "a1b2c3d4-0000-4000-8000-000000000001"
      },
      "usedVariables": [
        "_station"
      ],
      "text": "// Turbine options, narrowed by the selected stations.\nAssetMaster()\n| where Station in (_station) or isempty(_station)\n| distinct Turbine\n| sort by Turbine asc"
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000021",
      "dataSource": {
        "kind": "inline",
        "dataSourceId": "a1b2c3d4-0000-4000-8000-000000000001"
      },
      "usedVariables": [
        "_startTime",
        "_endTime",
        "_station",
        "_turbine"
      ],
      "text": "TelemetryEnriched(_startTime, _endTime, _station, _turbine)\n| summarize ['Readings'] = count()"
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000022",
      "dataSource": {
        "kind": "inline",
        "dataSourceId": "a1b2c3d4-0000-4000-8000-000000000001"
      },
      "usedVariables": [
        "_startTime",
        "_endTime",
        "_station",
        "_turbine"
      ],
      "text": "TelemetryEnriched(_startTime, _endTime, _station, _turbine)\n| summarize ['Turbines reporting'] = dcount(Turbine)"
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000023",
      "dataSource": {
        "kind": "inline",
        "dataSourceId": "a1b2c3d4-0000-4000-8000-000000000001"
      },
      "usedVariables": [
        "_startTime",
        "_endTime",
        "_station",
        "_turbine"
      ],
      "text": "TelemetryEnriched(_startTime, _endTime, _station, _turbine)\n| where SignalGroup =~ 'power'\n| summarize ['Avg power (MW)'] = round(avg(value), 1)"
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000024",
      "dataSource": {
        "kind": "inline",
        "dataSourceId": "a1b2c3d4-0000-4000-8000-000000000001"
      },
      "usedVariables": [
        "_startTime",
        "_endTime",
        "_station",
        "_turbine"
      ],
      "text": "TelemetryEnriched(_startTime, _endTime, _station, _turbine)\n| summarize Total = count(), Good = countif(quality =~ 'GOOD')\n| project ['Good quality %'] = iff(Total == 0, 0.0, round(100.0 * Good / Total, 1))"
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000031",
      "dataSource": {
        "kind": "inline",
        "dataSourceId": "a1b2c3d4-0000-4000-8000-000000000001"
      },
      "usedVariables": [
        "_startTime",
        "_endTime",
        "_station",
        "_turbine"
      ],
      "text": "let Bin = case(_endTime - _startTime > 7d, 1h, _endTime - _startTime > 1d, 15m, _endTime - _startTime > 6h, 5m, 30s);\nTelemetryEnriched(_startTime, _endTime, _station, _turbine)\n| where SignalGroup =~ 'power'\n| summarize Value = round(avg(value), 2) by Turbine, event_time = bin(event_time, Bin)\n| project event_time, Turbine, Value\n| sort by event_time asc\n| render timechart"
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000032",
      "dataSource": {
        "kind": "inline",
        "dataSourceId": "a1b2c3d4-0000-4000-8000-000000000001"
      },
      "usedVariables": [
        "_startTime",
        "_endTime",
        "_station",
        "_turbine"
      ],
      "text": "let Bin = case(_endTime - _startTime > 7d, 1h, _endTime - _startTime > 1d, 15m, _endTime - _startTime > 6h, 5m, 30s);\nTelemetryEnriched(_startTime, _endTime, _station, _turbine)\n| where SignalGroup =~ 'pressure'\n| summarize Value = round(avg(value), 2) by Turbine, event_time = bin(event_time, Bin)\n| project event_time, Turbine, Value\n| sort by event_time asc\n| render timechart"
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000033",
      "dataSource": {
        "kind": "inline",
        "dataSourceId": "a1b2c3d4-0000-4000-8000-000000000001"
      },
      "usedVariables": [
        "_startTime",
        "_endTime",
        "_station",
        "_turbine"
      ],
      "text": "let Bin = case(_endTime - _startTime > 7d, 1h, _endTime - _startTime > 1d, 15m, _endTime - _startTime > 6h, 5m, 30s);\nTelemetryEnriched(_startTime, _endTime, _station, _turbine)\n| where SignalGroup =~ 'speed'\n| summarize Value = round(avg(value), 2) by Turbine, event_time = bin(event_time, Bin)\n| project event_time, Turbine, Value\n| sort by event_time asc\n| render timechart"
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000034",
      "dataSource": {
        "kind": "inline",
        "dataSourceId": "a1b2c3d4-0000-4000-8000-000000000001"
      },
      "usedVariables": [
        "_startTime",
        "_endTime",
        "_station",
        "_turbine"
      ],
      "text": "let Bin = case(_endTime - _startTime > 7d, 1h, _endTime - _startTime > 1d, 15m, _endTime - _startTime > 6h, 5m, 30s);\nTelemetryEnriched(_startTime, _endTime, _station, _turbine)\n| where SignalGroup =~ 'temperature'\n| summarize Value = round(avg(value), 2) by Turbine, event_time = bin(event_time, Bin)\n| project event_time, Turbine, Value\n| sort by event_time asc\n| render timechart"
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000035",
      "dataSource": {
        "kind": "inline",
        "dataSourceId": "a1b2c3d4-0000-4000-8000-000000000001"
      },
      "usedVariables": [
        "_startTime",
        "_endTime",
        "_station",
        "_turbine"
      ],
      "text": "// Two vibration sensors per turbine, so the series key carries both.\nlet Bin = case(_endTime - _startTime > 7d, 1h, _endTime - _startTime > 1d, 15m, _endTime - _startTime > 6h, 5m, 30s);\nTelemetryEnriched(_startTime, _endTime, _station, _turbine)\n| where SignalGroup =~ 'vibration'\n| summarize Value = round(avg(value), 2) by Sensor = strcat(Turbine, ' ', Signal), event_time = bin(event_time, Bin)\n| project event_time, Sensor, Value\n| sort by event_time asc\n| render timechart"
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000036",
      "dataSource": {
        "kind": "inline",
        "dataSourceId": "a1b2c3d4-0000-4000-8000-000000000001"
      },
      "usedVariables": [
        "_startTime",
        "_endTime",
        "_station",
        "_turbine"
      ],
      "text": "TelemetryEnriched(_startTime, _endTime, _station, _turbine)\n| summarize arg_max(event_time, value, quality) by Station, Turbine, SignalGroup, Signal, Unit\n| project Station, Turbine, ['Group'] = SignalGroup, Signal, ['Last value'] = round(value, 2), Unit, ['Quality'] = quality, ['Last seen'] = event_time\n| sort by Station asc, Turbine asc, Signal asc"
    }
  ],
  "tiles": [
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000041",
      "title": "Readings",
      "pageId": "a1b2c3d4-0000-4000-8000-000000000002",
      "layout": {
        "x": 0,
        "y": 0,
        "width": 6,
        "height": 3
      },
      "queryRef": {
        "kind": "query",
        "queryId": "a1b2c3d4-0000-4000-8000-000000000021"
      },
      "visualType": "card",
      "visualOptions": {
        "multiStat__textSize": "large",
        "multiStat__valueColumn": "Readings",
        "multiStat__labelColumn": null,
        "colorRulesDisabled": true,
        "crossFilterDisabled": true,
        "drillthroughDisabled": true
      }
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000042",
      "title": "Turbines reporting",
      "pageId": "a1b2c3d4-0000-4000-8000-000000000002",
      "layout": {
        "x": 6,
        "y": 0,
        "width": 6,
        "height": 3
      },
      "queryRef": {
        "kind": "query",
        "queryId": "a1b2c3d4-0000-4000-8000-000000000022"
      },
      "visualType": "card",
      "visualOptions": {
        "multiStat__textSize": "large",
        "multiStat__valueColumn": "Turbines reporting",
        "multiStat__labelColumn": null,
        "colorRulesDisabled": true,
        "crossFilterDisabled": true,
        "drillthroughDisabled": true
      }
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000043",
      "title": "Avg power output (MW)",
      "pageId": "a1b2c3d4-0000-4000-8000-000000000002",
      "layout": {
        "x": 12,
        "y": 0,
        "width": 6,
        "height": 3
      },
      "queryRef": {
        "kind": "query",
        "queryId": "a1b2c3d4-0000-4000-8000-000000000023"
      },
      "visualType": "card",
      "visualOptions": {
        "multiStat__textSize": "large",
        "multiStat__valueColumn": "Avg power (MW)",
        "multiStat__labelColumn": null,
        "colorRulesDisabled": true,
        "crossFilterDisabled": true,
        "drillthroughDisabled": true
      }
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000044",
      "title": "Good quality %",
      "pageId": "a1b2c3d4-0000-4000-8000-000000000002",
      "layout": {
        "x": 18,
        "y": 0,
        "width": 6,
        "height": 3
      },
      "queryRef": {
        "kind": "query",
        "queryId": "a1b2c3d4-0000-4000-8000-000000000024"
      },
      "visualType": "card",
      "visualOptions": {
        "multiStat__textSize": "large",
        "multiStat__valueColumn": "Good quality %",
        "multiStat__labelColumn": null,
        "colorRulesDisabled": true,
        "crossFilterDisabled": true,
        "drillthroughDisabled": true
      }
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000045",
      "title": "Power output (MW)",
      "pageId": "a1b2c3d4-0000-4000-8000-000000000002",
      "layout": {
        "x": 0,
        "y": 3,
        "width": 12,
        "height": 8
      },
      "queryRef": {
        "kind": "query",
        "queryId": "a1b2c3d4-0000-4000-8000-000000000031"
      },
      "visualType": "timechart",
      "visualOptions": {
        "xColumn": "event_time",
        "yColumns": [
          "Value"
        ],
        "seriesColumns": [
          "Turbine"
        ],
        "xColumnTitle": "",
        "yColumnTitle": "MW",
        "hideLegend": false,
        "legendLocation": "right",
        "xAxisScale": "linear",
        "yAxisScale": "linear",
        "crossFilterDisabled": true,
        "drillthroughDisabled": true,
        "colorRulesDisabled": true
      }
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000046",
      "title": "Inlet pressure (bar)",
      "pageId": "a1b2c3d4-0000-4000-8000-000000000002",
      "layout": {
        "x": 12,
        "y": 3,
        "width": 12,
        "height": 8
      },
      "queryRef": {
        "kind": "query",
        "queryId": "a1b2c3d4-0000-4000-8000-000000000032"
      },
      "visualType": "timechart",
      "visualOptions": {
        "xColumn": "event_time",
        "yColumns": [
          "Value"
        ],
        "seriesColumns": [
          "Turbine"
        ],
        "xColumnTitle": "",
        "yColumnTitle": "bar",
        "hideLegend": false,
        "legendLocation": "right",
        "xAxisScale": "linear",
        "yAxisScale": "linear",
        "crossFilterDisabled": true,
        "drillthroughDisabled": true,
        "colorRulesDisabled": true
      }
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000047",
      "title": "Turbine speed (rpm)",
      "pageId": "a1b2c3d4-0000-4000-8000-000000000002",
      "layout": {
        "x": 0,
        "y": 11,
        "width": 12,
        "height": 8
      },
      "queryRef": {
        "kind": "query",
        "queryId": "a1b2c3d4-0000-4000-8000-000000000033"
      },
      "visualType": "timechart",
      "visualOptions": {
        "xColumn": "event_time",
        "yColumns": [
          "Value"
        ],
        "seriesColumns": [
          "Turbine"
        ],
        "xColumnTitle": "",
        "yColumnTitle": "rpm",
        "hideLegend": false,
        "legendLocation": "right",
        "xAxisScale": "linear",
        "yAxisScale": "linear",
        "crossFilterDisabled": true,
        "drillthroughDisabled": true,
        "colorRulesDisabled": true
      }
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000048",
      "title": "Turbine temperature (C)",
      "pageId": "a1b2c3d4-0000-4000-8000-000000000002",
      "layout": {
        "x": 12,
        "y": 11,
        "width": 12,
        "height": 8
      },
      "queryRef": {
        "kind": "query",
        "queryId": "a1b2c3d4-0000-4000-8000-000000000034"
      },
      "visualType": "timechart",
      "visualOptions": {
        "xColumn": "event_time",
        "yColumns": [
          "Value"
        ],
        "seriesColumns": [
          "Turbine"
        ],
        "xColumnTitle": "",
        "yColumnTitle": "C",
        "hideLegend": false,
        "legendLocation": "right",
        "xAxisScale": "linear",
        "yAxisScale": "linear",
        "crossFilterDisabled": true,
        "drillthroughDisabled": true,
        "colorRulesDisabled": true
      }
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000049",
      "title": "Vibration (mm/s)",
      "pageId": "a1b2c3d4-0000-4000-8000-000000000002",
      "layout": {
        "x": 0,
        "y": 19,
        "width": 12,
        "height": 8
      },
      "queryRef": {
        "kind": "query",
        "queryId": "a1b2c3d4-0000-4000-8000-000000000035"
      },
      "visualType": "timechart",
      "visualOptions": {
        "xColumn": "event_time",
        "yColumns": [
          "Value"
        ],
        "seriesColumns": [
          "Sensor"
        ],
        "xColumnTitle": "",
        "yColumnTitle": "mm/s",
        "hideLegend": false,
        "legendLocation": "right",
        "xAxisScale": "linear",
        "yAxisScale": "linear",
        "crossFilterDisabled": true,
        "drillthroughDisabled": true,
        "colorRulesDisabled": true,
        "selectedDataOnLoad": {
          "all": false,
          "limit": 10
        }
      }
    },
    {
      "id": "a1b2c3d4-0000-4000-8000-000000000050",
      "title": "Latest reading per signal",
      "pageId": "a1b2c3d4-0000-4000-8000-000000000002",
      "layout": {
        "x": 12,
        "y": 19,
        "width": 12,
        "height": 8
      },
      "queryRef": {
        "kind": "query",
        "queryId": "a1b2c3d4-0000-4000-8000-000000000036"
      },
      "visualType": "table",
      "visualOptions": {
        "crossFilterDisabled": true,
        "drillthroughDisabled": true,
        "colorRulesDisabled": true
      }
    }
  ]
}"""

print("Embedded seed:", len(DASHBOARD_SEED_JSON), "bytes")


In [ ]:
# =========================
# CELL 4 - Load the dashboard definition
# The FILE is the source of truth; the seed is only a bootstrap.
# =========================

definition_path = f"{ONELAKE_ROOT.rsplit('/Tables', 1)[0]}/{DASHBOARD_FILES_DIR}/{DASHBOARD_NAME}.json"
raw_definition = None
source_used = None

try:
    raw_definition = notebookutils.fs.head(definition_path, 20 * 1024 * 1024)
    source_used = definition_path
except Exception as exc:  # noqa: BLE001 - first run in a fresh workspace
    print("No definition in the lakehouse:", exc)

if not raw_definition and DASHBOARD_SOURCE_URL:
    resp = requests.get(DASHBOARD_SOURCE_URL, timeout=60)
    if resp.status_code == 200:
        raw_definition, source_used = resp.text, DASHBOARD_SOURCE_URL
    else:
        print(f"Source URL returned {resp.status_code}.")

if not raw_definition:
    raw_definition, source_used = DASHBOARD_SEED_JSON, "embedded seed (CELL 3)"

dashboard_def = json.loads(raw_definition)
print("Loaded definition from:", source_used)
print("  schema_version:", dashboard_def.get("schema_version"))
print("  tiles:", len(dashboard_def.get("tiles", [])), "| queries:", len(dashboard_def.get("queries", [])))
print("  parameters:", [p.get("displayName") for p in dashboard_def.get("parameters", [])])


In [ ]:
# =========================
# CELL 5 - Re-point the definition at THIS workspace and validate
# Handles both the placeholder seed and a file downloaded from another workspace.
# =========================

data_sources = dashboard_def.get("dataSources") or []
if not data_sources:
    raise RuntimeError("The definition has no dataSources entry.")

for ds in data_sources:
    ds["kind"] = "kusto-trident"
    ds["clusterUri"] = cluster_query_uri
    ds["databaseArtifactId"] = kql_db_id
    ds["database"] = kql_db_name
    ds["workspace"] = workspace_id
    if str(ds.get("name", "")).startswith("__") or not ds.get("name"):
        ds["name"] = kql_db_name
    print("data source ->", ds["name"], ds["clusterUri"], ds["database"])

resolved_json = json.dumps(dashboard_def, indent=2)
leftovers = [t for t in ("__CLUSTER_QUERY_URI__", "__KQL_DB_ID__", "__KQL_DB_NAME__", "__WORKSPACE_ID__") if t in resolved_json]
if leftovers:
    raise RuntimeError(f"Unresolved placeholders remain: {leftovers}")

# Every tile query must run before we deploy.
BINDINGS = {
    "_startTime": "let _startTime = ago(4h);",
    "_endTime": "let _endTime = now();",
    "_station": "let _station = dynamic(null);",
    "_turbine": "let _turbine = dynamic(null);",
}

failures = []
for query in dashboard_def["queries"]:
    prelude = "\n".join(BINDINGS[v] for v in query.get("usedVariables", []) if v in BINDINGS)
    try:
        cols, rows = kusto(f"{prelude}\n{query['text']}" if prelude else query["text"], endpoint="query")
        print(f"  [ok]   {query['id'][-4:]}  rows={len(rows):<6} cols={cols}")
    except Exception as exc:  # noqa: BLE001
        failures.append(query["id"])
        print(f"  [FAIL] {query['id'][-4:]}  {exc}")

if failures:
    raise RuntimeError(f"{len(failures)} dashboard queries failed; not deploying. See errors above.")
print("\nAll dashboard queries validated.")


In [ ]:
# =========================
# CELL 6 - Deploy as a Fabric KQLDashboard item
# =========================

def b64(text):
    return base64.b64encode(text.encode("utf-8")).decode("utf-8")


def wait_for_lro(resp, max_tries=40, delay_sec=5):
    location = resp.headers.get("Location")
    if not location:
        return resp
    for attempt in range(1, max_tries + 1):
        poll = requests.get(location, headers=FABRIC_HEADERS)
        status = poll.json().get("status") if poll.content else None
        if status in ("Succeeded", "Completed"):
            return poll
        if status == "Failed":
            raise RuntimeError(f"Operation failed: {poll.text[:800]}")
        print(f"  waiting ({attempt}/{max_tries}, status={status})...")
        time.sleep(delay_sec)
    raise RuntimeError("Operation did not complete in time.")


platform = {
    "$schema": "https://developer.microsoft.com/json-schemas/fabric/gitIntegration/platformProperties/2.0.0/schema.json",
    "metadata": {"type": "KQLDashboard", "displayName": DASHBOARD_NAME},
    "config": {"version": "2.0", "logicalId": str(uuid.uuid4())},
}
definition = {
    "parts": [
        {"path": "RealTimeDashboard.json", "payload": b64(resolved_json), "payloadType": "InlineBase64"},
        {"path": ".platform", "payload": b64(json.dumps(platform)), "payloadType": "InlineBase64"},
    ]
}

existing = next(
    (i for i in list_items(workspace_id)
     if i.get("displayName") == DASHBOARD_NAME and i.get("type", "").lower() == "kqldashboard"),
    None,
)

if existing:
    dashboard_item_id = existing["id"]
    resp = requests.post(
        f"{FABRIC_BASE_URL}/workspaces/{workspace_id}/items/{dashboard_item_id}/updateDefinition",
        headers=FABRIC_HEADERS,
        json={"definition": definition},
    )
    if resp.status_code not in (200, 202):
        raise RuntimeError(f"updateDefinition failed ({resp.status_code}): {resp.text[:800]}")
    wait_for_lro(resp)
    print(f"Updated existing KQLDashboard '{DASHBOARD_NAME}' ({dashboard_item_id}).")
else:
    payload = {"displayName": DASHBOARD_NAME, "type": "KQLDashboard", "definition": definition}
    if target_folder_id:
        payload["folderId"] = target_folder_id
    resp = requests.post(f"{FABRIC_BASE_URL}/workspaces/{workspace_id}/items", headers=FABRIC_HEADERS, json=payload)
    if resp.status_code not in (200, 201, 202):
        raise RuntimeError(f"create KQLDashboard failed ({resp.status_code}): {resp.text[:800]}")
    if resp.status_code == 202:
        body = wait_for_lro(resp).json()
        dashboard_item_id = body.get("id") or (body.get("resourceLocation") or "").rsplit("/", 1)[-1]
    else:
        dashboard_item_id = resp.json().get("id")
    print(f"Created KQLDashboard '{DASHBOARD_NAME}' ({dashboard_item_id}).")

print(f"\nOpen it: https://app.fabric.microsoft.com/groups/{workspace_id}/kustodashboards/{dashboard_item_id}")


In [ ]:
# =========================
# CELL 7 - Write the resolved copy back to Files/ and record the item id
# The written file is import-ready: Fabric UI -> Manage -> Replace with file.
# =========================

try:
    notebookutils.fs.mkdirs(definition_path.rsplit("/", 1)[0])
except Exception:  # noqa: BLE001 - already exists
    pass

notebookutils.fs.put(definition_path, resolved_json, True)
print("Resolved definition written to:", definition_path)

try:
    from delta.tables import DeltaTable
    from pyspark.sql import functions as F

    persist = {
        "basic_dashboard_name": DASHBOARD_NAME,
        "basic_dashboard_id": dashboard_item_id,
        "basic_dashboard_definition_path": definition_path,
    }
    persist_df = (
        spark.createDataFrame([{"setting_name": k, "setting_value": str(v)} for k, v in persist.items()])
        .withColumn("updated_utc", F.current_timestamp())
    )
    (
        DeltaTable.forName(spark, SETTINGS_TABLE).alias("target")
        .merge(persist_df.alias("source"), "target.setting_name = source.setting_name")
        .whenMatchedUpdate(set={"setting_value": "source.setting_value", "updated_utc": "source.updated_utc"})
        .whenNotMatchedInsert(values={
            "setting_name": "source.setting_name",
            "setting_value": "source.setting_value",
            "updated_utc": "source.updated_utc",
        })
        .execute()
    )
    print("Persisted to", SETTINGS_TABLE, "->", persist)
except Exception as exc:  # noqa: BLE001 - standalone run without the settings table
    print("Skipped persisting to", SETTINGS_TABLE, "-", exc)

print("\nDone.")
